# Loading Glider data and convert to OG1 format

This notebook is dedicated to **load oceanographic glider datasets** from the University of Washington Seaglider program. The data corresponds to missions conducted near the Faroe Islands and are publicly available from NOAA’s National Centers for Environmental Information (NCEI).

---

## Glider Missions

- **Overview:** [Seaglider Faroes Missions](https://iop.apl.washington.edu/seaglider/index.php?mission=Faroes)  
- **Data Server:** [NOAA NCEI Seaglider Archive](https://www.ncei.noaa.gov/data/oceans/glider/seaglider/uw/)

The missions span from 2006 to 2009 and include multiple gliders (e.g., 005, 016, 101) and mission deployments. Below is a detailed breakdown of available missions and folder identifiers:

### November 2006
- 016 early recovered (124 dives) — `016/20061112` bad bad resolution data
- 101 full mission (578 dives) — `101/20061112` bad resolution of data
- 102 full mission (630 dives) — `102/20061112`

### February 2007
- 103 full mission (679 dives) — `103/20070218`
- 105 FSC section (125 dives) — _no folder_
- 104 early recovery (70 dives) — _no folder_

### June 2007
- 101 full mission (453 dives) — `101/20070609`
- 102 early recovery (22 dives) — _no folder_

### September 2007
- 012 partial mission (188 dives) — `012/20070831`
- 104 full mission (371 dives) — `104/20070901`
- 105 full mission (373 dives) — _no folder_

### November 2007
- 016 (452 dives) — `016/20071113`
- 102 (454 dives) — `102/20071113`
- 103 (524 dives) — `103/20071113`

### February 2008
- 014 (70 dives) — `014/20080214`   not at IFR
- 104 (112 dives) — `104/20080214`

### June 2008
- 102 (5 dives) — _no folder_
- 005 (386 dives) — `005/20080606`
- 016 (421 dives) — `016/20080607`

### August 2008
- 105 (13 dives) — _no folder_
- 103 (10 dives) — _no folder_
- 014 (439 dives) — `014/20080222` (_folder name misleading_)

### November 2008
- 104 (1 dive) — _no folder_  
- 101 (338 dives) — `101/20081108`
- 005 (486 dives) — `005/20081106`

### February 2009
- 103 (481 dives) — `103/20090223`
- 104 (4 dives) — _no folder_

### June 2009
- 016 (297 dives) — `016/20090605`
- 105 (346 dives) — _no folder_
- 005 (464 dives) — `005/20090829`

---
All missions that cannot be used for the analysis (bad data or not in the region) are not considered in the glider_server.yaml

In [79]:
import pathlib
import sys
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from seagliderOG1 import convertOG1, writers
from tqdm import tqdm
import xarray as xr
import importlib
import numpy as np
import matplotlib.dates as mdates

script_dir = pathlib.Path().parent.absolute()
parent_dir = script_dir.parents[0]
sys.path.append(str(parent_dir))
from dissipationSML import reading, tools, interactive, utilities

In [80]:
importlib.reload(reading)
importlib.reload(tools)
importlib.reload(interactive)
plotting_style = '/Users/tillmoritz/Desktop/Master_thesis/Git/dissipationSML/dissipationSML/plotting.mplstyle'

# 1. Data Loading and Conversion

### 1.1 Choose glider mission

In [ ]:
# Example usage
yaml_path = str(parent_dir) + '/dissipationSML/glider_server.yaml'  # Update with your actual path
selected_glider = interactive.interactive_glider_selection(yaml_path)

Dropdown(description='Select Glider:', options=('005', '012', '014', '016', '101', '102', '103', '104', '105')…

Dropdown(description='Select Mission:', options=('06/08 (dives: 386)', '08/09 (dives: 464)', '11/08 (dives: 48…

Button(description='Confirm Selection', style=ButtonStyle())

Selected Path: https://www.ncei.noaa.gov/data/oceans/glider/seaglider/uw/005/20080606/


In [82]:
### From the chosen glider mission, the important variables are taken. The destination folder is also selected here.
### The mission path gives the exact path to the dedicated mission of the chosen glider. It will be used later to save and load the concatenated/converted data
data_path = selected_glider['path']
end_profile = selected_glider['dives']
destination_folder = "/Users/tillmoritz/Desktop/Master_thesis/Data/Glider"
mission_path = destination_folder + '/' + selected_glider['glider'] + '/' + selected_glider['mission']

### 1.2 (Down-)loading the data into a list of .nc files and then convert into OG1 format. 
The list of datasets is concatented into one dataset and then saved at the mission path 

In [61]:
### All datasets from the glider mission are downloaded and then loaded as .nc file into one list
datasets = reading.read_basestation(data_path, destination_path=destination_folder, start_profile=1, end_profile=end_profile)


File already exists: /Users/tillmoritz/Desktop/Master_thesis/Data/Glider/005/20080606/p0050001_20080606.nc
File already exists: /Users/tillmoritz/Desktop/Master_thesis/Data/Glider/005/20080606/p0050002_20080606.nc
File already exists: /Users/tillmoritz/Desktop/Master_thesis/Data/Glider/005/20080606/p0050003_20080606.nc
File already exists: /Users/tillmoritz/Desktop/Master_thesis/Data/Glider/005/20080606/p0050004_20080607.nc
File already exists: /Users/tillmoritz/Desktop/Master_thesis/Data/Glider/005/20080606/p0050005_20080607.nc
File already exists: /Users/tillmoritz/Desktop/Master_thesis/Data/Glider/005/20080606/p0050006_20080607.nc
File already exists: /Users/tillmoritz/Desktop/Master_thesis/Data/Glider/005/20080606/p0050007_20080607.nc
File already exists: /Users/tillmoritz/Desktop/Master_thesis/Data/Glider/005/20080606/p0050008_20080607.nc
File already exists: /Users/tillmoritz/Desktop/Master_thesis/Data/Glider/005/20080606/p0050009_20080607.nc
File already exists: /Users/tillmorit

In [62]:
variables_needed = ['TIME','LONGITUDE','LATITUDE','PROFILE_NUMBER','divenum', # measurment point specific
                    'PITCH', 'ROLL','HEADING',
                    'DEPTH','PRES',
                    'GLIDER_VERT_VELO_MODEL','GLIDER_HORZ_VELO_MODEL','GLIDE_SPEED','GLIDE_SPEED_QC', # vertical and horizontal velocity with quality control
                    'TEMP','TEMP_QC','PSAL','PSAL_QC','PSAL_RAW','PSAL_RAW_QC','TEMP_RAW','TEMP_RAW_QC', # CTD data with extra quality control
                    'SIGTHETA','SIGMA_T','THETA'] # potential density and temperature

In [63]:
### convert the dataset to OG1 format only with the variables needed
ds = reading.convert_with_variables(datasets,variables_needed)

Converting datasets to OG1 format:  24%|██▍       | 94/386 [00:41<02:06,  2.31it/s]/Users/tillmoritz/miniconda3/envs/dissipationSML/lib/python3.13/site-packages/seagliderOG1/utilities.py:35: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  ds1 = ds1.assign_coords(longitude=("sg_data_point", [float('nan')] * ds1.dims['sg_data_point']))
/Users/tillmoritz/miniconda3/envs/dissipationSML/lib/python3.13/site-packages/seagliderOG1/utilities.py:38: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  ds1 = ds1.assign_coords(latitude=("sg_data_point", [float('nan')] * ds1.dims['sg_data_point']))


p0050095_20080626: No coord longitude - adding as NaNs to length of sg_data_point
p0050095_20080626: No coord latitude - adding as NaNs to length of sg_data_point
p0050095_20080626: !!! No variable ctd_time - returning an empty dataset


Converting datasets to OG1 format: 100%|██████████| 386/386 [02:58<00:00,  2.17it/s]


In [65]:
### add gradients of temperature and salinity to the dataset for spike control
ds['dTdz'] = ds['TEMP'].differentiate('DEPTH')
ds['dSdz'] = ds['PSAL'].differentiate('DEPTH')
#ds['PSAL'].differentiate('DEPTH')

/Users/tillmoritz/miniconda3/envs/dissipationSML/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:1307: RuntimeWarning: divide by zero encountered in divide
  a = -(dx2)/(dx1 * (dx1 + dx2))
/Users/tillmoritz/miniconda3/envs/dissipationSML/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:1307: RuntimeWarning: invalid value encountered in divide
  a = -(dx2)/(dx1 * (dx1 + dx2))
/Users/tillmoritz/miniconda3/envs/dissipationSML/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:1308: RuntimeWarning: divide by zero encountered in divide
  b = (dx2 - dx1) / (dx1 * dx2)
/Users/tillmoritz/miniconda3/envs/dissipationSML/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:1308: RuntimeWarning: invalid value encountered in divide
  b = (dx2 - dx1) / (dx1 * dx2)
/Users/tillmoritz/miniconda3/envs/dissipationSML/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:1309: RuntimeWarning: divide by zero encountered in divide
  c = dx1 / (dx2 * (dx

In [66]:
interactive.interactive_profile(ds)

Output()

In [67]:
import pandas as pd
def remove_spikes(ds, var, window=20, n_std=4, grad_threshold=1.0):
    """
    Remove spikes from a single variable in an xarray Dataset using
    a rolling mean / standard-deviation filter and a vertical gradient check.
    Only the breakout point of a spike is removed, not the following recovery point.
    """

    ds = ds.copy()  # avoid modifying input in place

    arr = ds[var].values.astype(float)   # ensure float dtype
    depth = ds['DEPTH'].values

    # ---- Safe manual gradient (handles repeated depths) ----
    dz = np.diff(depth, prepend=depth[0])
    dz_safe = dz.copy()
    dz_safe[dz_safe == 0] = np.nan       # avoid divide-by-zero

    dvar = np.diff(arr, prepend=arr[0])
    grad_arr = dvar / dz_safe            # safe gradient

    # ---- Rolling mean/std using pandas ----
    s = pd.Series(arr)
    roll_mean = s.rolling(window=window, center=True, min_periods=1).mean()
    roll_std  = s.rolling(window=window, center=True, min_periods=1).std()

    deviation_mask = np.abs(s - roll_mean) > (n_std * roll_std)

    # ---- Gradient spike detection (only breakout point) ----
    grad_spike_mask = np.zeros_like(arr, dtype=bool)

    mask_pair = (
        (np.abs(grad_arr[:-1]) > grad_threshold) &
        (np.abs(grad_arr[1:]) > grad_threshold) &
        (np.sign(grad_arr[:-1]) != np.sign(grad_arr[1:]))
    )

    grad_spike_mask[:-1] = mask_pair     # spike at index i only

    # ---- Combine masks ----
    spike_mask = deviation_mask | grad_spike_mask

    # ---- Replace spikes with NaN ----
    s[spike_mask] = np.nan
    ds[var].values = s.values

    return ds


In [68]:
importlib.reload(tools)
### Remove spikes in temperature, salinity before calculating density
ds = remove_spikes(ds, 'PSAL',window=20 ,n_std=3,grad_threshold=1)
#ds = tools.remove_spikes(ds, 'TEMP',window=30 ,n_std=3,grad_threshold=0.5)

In [69]:
### calculate potential density and add it to the dataset
ds = tools.add_densities(ds)


### Interactively checking the data for different profiles
- Focus especially on the qualitity of the modelled vertical velocity
- Use pitch boundaries for identifying bad data

Questions:
- Should the calculation of vertical velocity be dependent also on other variables? roll?
- Does the flight model actually work that well? - Dependency on the pitch in the profiles? 

In [56]:
### check the data 
interactive.interactive_profile(ds)

Output()

In [70]:
ds = tools.add_vertical_water_velocity(ds, pitch_min = 10, pitch_max = 20)

In [71]:
ds

<xarray.Dataset> Size: 52MB
Dimensions:         (N_MEASUREMENTS: 332364)
Coordinates:
    TIME            (N_MEASUREMENTS) datetime64[ns] 3MB 2008-06-06T18:02:56 ....
    LONGITUDE       (N_MEASUREMENTS) float64 3MB -8.274 -8.274 ... -6.049 -6.048
    LATITUDE        (N_MEASUREMENTS) float64 3MB 61.42 61.41 ... 63.69 63.7
    DEPTH           (N_MEASUREMENTS) float64 3MB 0.0 0.0 0.387 ... -0.1072 0.0
Dimensions without coordinates: N_MEASUREMENTS
Data variables: (12/26)
    PROFILE_NUMBER  (N_MEASUREMENTS) float64 3MB 1.0 1.0 1.0 ... 772.0 772.0
    DIVE_NUMBER     (N_MEASUREMENTS) int32 1MB 1 1 1 1 1 ... 386 386 386 386 386
    PITCH           (N_MEASUREMENTS) float32 1MB nan nan -67.3 ... 18.9 4.4 nan
    ROLL            (N_MEASUREMENTS) float32 1MB nan nan -5.7 ... -1.8 -5.5 nan
    HEADING         (N_MEASUREMENTS) float32 1MB nan nan 354.0 ... 267.3 nan
    PRES            (N_MEASUREMENTS) float32 1MB nan nan 0.926 ... 0.05033 nan
    ...              ...
    THETA           (N_MEASUREMENTS) float32 1MB nan nan nan ... 11.34 nan nan
    dTdz            (N_MEASUREMENTS) float32 1MB nan nan nan ... nan nan
    dSdz            (N_MEASUREMENTS) float32 1MB nan nan nan nan ... nan nan nan
    SIGMA_1         (N_MEASUREMENTS) float64 3MB nan nan nan ... 31.26 nan nan
    W_MEASURED      (N_MEASUREMENTS) float64 3MB nan -0.1202 ... 0.1548 nan
    W_WATER         (N_MEASUREMENTS) float64 3MB nan nan nan nan ... nan nan nan
Attributes: (12/40)
    title:                                      OceanGliders trajectory file
    id:                                         sg005_20080606T180738_delayed
    platform:                                   sub-surface gliders
    platform_vocabulary:                        https://vocab.nerc.ac.uk/coll...
    naming_authority:                           edu.washington.apl
    institution:                                School of Oceanography\nUnive...
    ...                                         ...
    license:                                    These data may be redistribut...
    acknowledgment:                             National Science Foundation, ...
    keywords:                                   Water Temperature, Conductivi...
    disclaimer:                                 Data provided AS-IS.
    file_version:                               2.71
    pitch_range:                                10 to 20 degrees

In [72]:
### cut the dataset to the time period of interest
mask = (ds['TIME'] > np.datetime64('2006-01-01')) & (ds['TIME'] < np.datetime64('2009-12-31'))
ds = ds.where(mask, drop=True)

In [73]:
ds.attrs['Glider'] = selected_glider['glider']
ds.attrs['Mission'] = selected_glider['mission']

In [75]:
import pathlib

file_name = "all_data_OG1.nc"
dataset_path = pathlib.Path(mission_path) / file_name
remove_old = False

# Check first if the dataset already exists to avoid overwriting
if not dataset_path.exists():
    writers.save_dataset(ds, dataset_path)
else:
    if remove_old:
        dataset_path.unlink()  # remove the old file
        writers.save_dataset(ds, dataset_path)
    else:
        print("Dataset already exists at the specified location. No overwriting.")

### Load the saved dataset from the mission path

In [83]:
ds = xr.open_dataset(dataset_path)

In [84]:
ds

<xarray.Dataset> Size: 53MB
Dimensions:         (N_MEASUREMENTS: 332364)
Coordinates:
    TIME            (N_MEASUREMENTS) datetime64[ns] 3MB ...
    LONGITUDE       (N_MEASUREMENTS) float64 3MB ...
    LATITUDE        (N_MEASUREMENTS) float64 3MB ...
    DEPTH           (N_MEASUREMENTS) float64 3MB ...
Dimensions without coordinates: N_MEASUREMENTS
Data variables: (12/26)
    PROFILE_NUMBER  (N_MEASUREMENTS) float64 3MB ...
    DIVE_NUMBER     (N_MEASUREMENTS) float64 3MB ...
    PITCH           (N_MEASUREMENTS) float32 1MB ...
    ROLL            (N_MEASUREMENTS) float32 1MB ...
    HEADING         (N_MEASUREMENTS) float32 1MB ...
    PRES            (N_MEASUREMENTS) float32 1MB ...
    ...              ...
    THETA           (N_MEASUREMENTS) float32 1MB ...
    dTdz            (N_MEASUREMENTS) float32 1MB ...
    dSdz            (N_MEASUREMENTS) float32 1MB ...
    SIGMA_1         (N_MEASUREMENTS) float64 3MB ...
    W_MEASURED      (N_MEASUREMENTS) float64 3MB ...
    W_WATER         (N_MEASUREMENTS) float64 3MB ...
Attributes: (12/42)
    title:                                      OceanGliders trajectory file
    id:                                         sg005_20080606T180738_delayed
    platform:                                   sub-surface gliders
    platform_vocabulary:                        https://vocab.nerc.ac.uk/coll...
    naming_authority:                           edu.washington.apl
    institution:                                School of Oceanography\nUnive...
    ...                                         ...
    keywords:                                   Water Temperature, Conductivi...
    disclaimer:                                 Data provided AS-IS.
    file_version:                               2.71
    pitch_range:                                10 to 20 degrees
    Glider:                                     005
    Mission:                                    20080606